##### Import libraries

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import unicodedata
import spacy
from spacy.matcher import PhraseMatcher, Matcher
# from spacy.tokens import Token

pd.set_option("display.max_columns", None)   # no column truncation
pd.set_option("display.max_rows", None)      # no row truncation
pd.set_option("display.width", None)         # no line-wrapping
pd.set_option("display.max_colwidth", None)

### Configuration and Functions used in the notebook

##### ACE Terms dictionary

Moved to a json file. Load it at the beginning of the annotation pipeline.

##### Regex configuration

In [2]:
# Cues in constant improvement, according to needs.

# NEGATION CUES
NEG_CUES = re.compile(
    r"(?:"
    # Specific multi-token negation constructions
    r"\b(?:no|sin)\s+antecedentes\s+de\b|"
    r"\b(?:no|sin)\s+historia\s+de\b|"
    r"\b(?:no|sin)\s+evidencia\s+de\b|"

    # Structured negation with verbs
    r"\bno\s+(?:refiere|presenta|consta|hay|ha|han|tuvo|tiene|presentó|refirió)\b|"

    # Verb-based negation
    r"\bniega(?:n)?\b|"
    r"\bdescarta(?:n)?\b|"

    # Generic negation
    r"\bsin\b"
    r")",
    re.I
)

# CHILDHOOD CUES
CHILD_CUES = re.compile(
    r"\b(?:"
    # Life stages
    r"(?:en|desde|durante)\s+(?:la|su)\s+"
    r"(?:infancia|niñez|adolescencia|infancia\s+temprana|infancia\s+precoz)|"

    # Childhood/adolescence without rigid article patterns
    r"en\s+(?:su\s+)?(?:infancia|niñez|adolescencia)|"
    r"durante\s+(?:la|su)\s+(?:infancia|niñez|adolescencia)|"
    r"desde\s+(?:la|su)\s+(?:infancia|niñez)|"

    # When he/she was a child / minor
    r"cuando\s+era\s+(?:niñ[oa]|menor|adolescente|pequeñ[oa])|"
    r"siendo\s+(?:niñ[oa]|menor|adolescente|pequeñ[oa])|"
    r"de\s+(?:niñ[oa]|pequeñ[oa])|"
    r"desde\s+(?:niñ[oa]|pequeñ[oa])|"

    # Explicit minor
    r"menor\s+de\s+edad|"
    r"siendo\s+menor|"
    r"cuando\s+era\s+menor\s+de\s+edad|"

    # Explicit age 0-17
    r"(?:a\s+los|con)\s+(?:[0-9]|1[0-7])\s*(?:a[nñ]os?|a\.?)(?:\s+de\s+edad)?|"
    r"cuando\s+(?:\w+\s+){0,3}?ten[ií]a\s+(?:[0-9]|1[0-7])\s*(?:a[nñ]os?|a\.?)(?:\s+de\s+edad)?|"

    # School-age / underage variants
    r"en\s+edad\s+escolar|"
    r"siendo\s+adolescente|"
    r"cuando\s+era\s+adolescente"
    r")\b",
    re.I
)

# Introduced to negate the existence of childhood ACEs, try with it
ADULT_CUES = re.compile(
    r"\b(?:"
    r"en\s+la\s+edad\s+adulta|"
    r"en\s+edad\s+adulta|"
    r"siendo\s+adult[oa]|"
    r"cuando\s+era\s+adult[oa]|"
    r"(?:a\s+los|con)\s+(?:1[89]|[2-9]\d)\s*(?:a[nñ]os?|a\.?)|"
    r"cuando\s+ten[ií]a\s+(?:1[89]|[2-9]\d)\s*(?:a[nñ]os?|a\.?)"
    r")\b",
    re.I
)

# TEMPORAL CUES
PAST_TIME_CUES = re.compile(
    r"\b(?:"
    r"(?<!desde\s)hace\s+\d+\s+(?:a[nñ]os|mes(?:es)?|semanas|d[ií]as|horas)"
    r")\b",
    re.I
)

RECENT_CUES = re.compile(
    r"\b(?:"
    r"recientemente|"
    r"en\s+los\s+[uú]ltimos\s+\d+\s+(?:d[ií]as|semanas|mes(?:es)?)|"
    r"esta\s+semana|"
    r"este\s+mes|"
    r"este\s+a[nñ]o"
    r")\b",
    re.I
)

CURRENT_CUES = re.compile(
    r"\b(?:"
    r"actualmente|"
    r"en\s+la\s+actualidad|"
    r"a\s+d[ií]a\s+de\s+hoy|"
    r"en\s+este\s+momento|"
    r"en\s+el\s+momento\s+actual|"
    r"hoy\s+en\s+d[ií]a"
    r")\b",
    re.I
)

ONGOING_CUES = re.compile(
    r"\b(?:"
    r"desde\s+hace\s+\d+\s+(?:a[nñ]os|mes(?:es)?|semanas|d[ií]as|horas)|"
    r"todav[ií]a|"
    r"a[uú]n|"
    r"contin[uú]a|contin[uú]an|"
    r"persiste|persisten|"
    r"sigue|siguen"
    r")\b",
    re.I
)

# DURATION CUES - > in third annotator version, removed and
# integrated them into ongoing. Reason: not enough support in dataset
# FREQUENCY CUES
FREQUENCY_CUES = re.compile(
    r"\b("
    r"en\s+una\s+(?:sola\s+|[uú]nica\s+)?ocasi[oó]n|"
    r"(?:solo|solamente)?\s*una\s+vez|"
    r"en\s+varias\s+ocasiones|"
    r"en\s+m[uú]ltiples\s+ocasiones|"
    r"en\s+repetidas\s+ocasiones|"
    r"en\s+repetidas\s+oportunidades?|"
    r"en\s+m[aá]s\s+de\s+una\s+ocasi[oó]n|"
    r"varias\s+veces|"
    r"muchas\s+veces|"
    r"m[aá]s\s+de\s+una\s+vez|"
    r"repetidamente|"
    r"de\s+forma\s+repetida|"
    r"de\s+manera\s+repetida|"
    r"de\s+forma\s+recurrente|"
    r"de\s+manera\s+recurrente|"
    r"recurrentemente|"
    r"ocasionalmente|ocasional"
    r")\b",
    re.I
)

# PERPETRATOR CUES
# Links btw spans
PERPETRATOR_LINK_CUES = re.compile(
    r"\b(?:"
    r"por|"
    r"por\s+parte\s+de(?:l|la|los|las|su|sus)?|"
    r"a\s+manos\s+de(?:l|la|los|las|su|sus)?|"
    r"ejercid[oa]\s+por|"
    r"causad[oa]\s+por|"
    r"cometid[oa]\s+por|"
    r"perpetrad[oa]\s+por|"
    r"de(?:l|la|los|las|su|sus)"   # 
    r")\b",
    re.I
)

# Perpetrator entity/role cues
PERPETRATOR_CUES = re.compile(
    r"\b(?:"
    r"padre\s+de\s+su\s+hij[oa]|"
    r"madre\s+de\s+su\s+hij[oa]|"
    r"marido\s+de\s+t[ií]a|"
    r"padre|madre|"
    r"herman[oa]|"
    r"t[ií][oa]|"
    r"abuel[oa]|"
    r"prim[oa]|"
    r"amig[oa]|"
    r"vecin[oa]|"
    r"compa[nñ]er[oa]|"
    r"conocid[oa]|"
    r"desconocid[oa]|"
    r"pareja|"
    r"ex(?:\s+pareja)?|"
    r"novi[oa]|"
    r"marido|mujer|"
    r"espos[oa]|"
    r"profesor|profesora|"
    r"maestr[oa]"
    r")\b",
    re.I
)

# CONTROL OF UNWANTED COMBINATIONS WITH "abandono" or any other generic ACE related term
ABANDONO_SUBSTANCE_CONTEXT = re.compile(
    r"\b(?:consumo|alcohol|tabaco|cannabis|t[óo]xicos|drogas|sustancias)\b",
    re.I
)

In [49]:
"""tests = [
    "No antecedentes de ",
    "sin antecedentes de ",
    "No historia de ",
    "sin evidencia de ",
    "no refiere ",
    "niega ",
    "descarta ",
    "sin ",
]

for t in tests:
    m = NEG_CUES.search(t)
    print(repr(t), "->", m.group(0) if m else None)"""

'tests = [\n    "No antecedentes de ",\n    "sin antecedentes de ",\n    "No historia de ",\n    "sin evidencia de ",\n    "no refiere ",\n    "niega ",\n    "descarta ",\n    "sin ",\n]\n\nfor t in tests:\n    m = NEG_CUES.search(t)\n    print(repr(t), "->", m.group(0) if m else None)'

In [4]:
# Possible separations between clauses in a sentence
# Need to differenciate according to the type of cue: perpetretor or temporal?
PERPETRATOR_BREAKS = re.compile(r"[.;:]", re.I)

GENERAL_CLAUSE_BREAKS = re.compile(
    r"[,:;]|\b(?:pero|aunque|sin embargo)\b",
    re.I
)
# Same with temporal cues, to include the stated verbs as detection of start of a sentence (due to the fact that there are
# EHRs with no full stops between sentences).
"""TEMPORAL_BREAKS = re.compile(
    r"[,:;]"
    r"|\b(?:pero|aunque|sin embargo)\b"
    r"|\b(?:refiere|niega|presenta|comenta|describe|manifiesta|explica|añade|indica|señala)\b",
    re.I
)"""
# Remove the comma as it makes lose some cues
TEMPORAL_BREAKS = re.compile(
    r"[;:]"
    r"|\b(?:pero|aunque|sin embargo)\b"
    r"|\b(?:refiere|niega|presenta|comenta|describe|manifiesta|explica|añade|indica|señala)\b",
    re.I
)


In [5]:
# To detect some associations, at post-processing.Ex: "maltrato físico y psicológico" implies "maltrato físico" and 
# "maltrato psicológico". Generalized to include "abuso" as well, and to be more flexible with the cues (ex: "maltrato físico, psicológico y verbal")
COORD_CUES = re.compile(
    r"\b(maltrato|abuso)\s+"                                   # group(1): shared head
    r"(f[ií]sic[oa]|psicol[oó]gic[oa]|emocional|verbal)"       # group(2): first modifier
    r"\s*(?:,)?\s*y\s+"
    r"(f[ií]sic[oa]|psicol[oó]gic[oa]|emocional|verbal)\b",    # group(3): second modifier
    re.I
)

COORD_TYPE_TO_CONCEPT = {
    "fisico": "ACE_PHYSICAL_ABUSE",
    "psicologico": "ACE_EMOTIONAL_ABUSE",
    "emocional": "ACE_EMOTIONAL_ABUSE",
    "verbal": "ACE_EMOTIONAL_ABUSE",
}

PARENTAL_ADJ_CUES = re.compile(r"\b(patern[oa]|matern[oa])\b", re.I)

In [6]:
# Modified in the third annotator version to fully align with the ACE term dict
SPECIFIC_ACE_CONCEPTS = {
    "ACE_SEXUAL_ABUSE",
    "ACE_ATTEMPTED_ABUSE",
    "ACE_PHYSICAL_ABUSE",
    "ACE_EMOTIONAL_ABUSE",
    "ACE_PHYSICAL_NEGLECT",
    "ACE_EMOTIONAL_NEGLECT",
    "ACE_UNSPECIFIED_NEGLECT",
    "ACE_DOMESTIC_VIOLENCE_EXPOSURE",
    "ACE_PARENT_SUBSTANCE",
    "ACE_PARENT_MENTAL_ILLNESS",
    "ACE_PARENT_INCARCERATION",
    "ACE_PARENT_DIVORCE_SEPARATION",
    "ACE_POVERTY_UNEMPLOYMENT",
}

In [7]:
# Modified in the third annotator version to fully align with the ACE term dict
CONCEPT_PRIORITY = {
    "ACE_ATTEMPTED_ABUSE": 5,

    "ACE_SEXUAL_ABUSE": 4,
    "ACE_PHYSICAL_ABUSE": 4,
    "ACE_EMOTIONAL_ABUSE": 4,
    "ACE_PHYSICAL_NEGLECT": 4,
    "ACE_EMOTIONAL_NEGLECT": 4,
    "ACE_UNSPECIFIED_NEGLECT": 4,
    "ACE_DOMESTIC_VIOLENCE_EXPOSURE": 4,
    "ACE_PARENT_SUBSTANCE": 4,
    "ACE_PARENT_MENTAL_ILLNESS": 4,
    "ACE_PARENT_INCARCERATION": 4,
    "ACE_PARENT_DIVORCE_SEPARATION": 4,
    "ACE_POVERTY_UNEMPLOYMENT": 4,

    "ACE_GENERAL_TRAUMA": 1,
}

##### Regex-based annotator functions with offsets for attributes on top of ACE terms

We modify the annotator functions that cut the spans in order to recover the offset of the start of the span. The initial annotator functions only kept the offset (start, end) for the ACE terms.
We want to keep them also for all the attribute cues, to help the BRAT annotation and, when the moment arrives, the transformer.

Summary of functions:
- 1. cut_left_at_break_with_offset
- 2. cut_right_at_break_with_offset
- 3. cut_right_for_perpetrator_with_offset
- 4. def strip_accents_keep_enye
- 5. normalize_perpetrator
- 6. negation_flag_local
- 7. temporal_flags_local
- 8. frequency_flags_local
- 9. perpetrator_flags_local
- 10. discard_ambiguous_ace_match
- 11. extract_ace_terms
- 12. add_coordination_derived_annotations
- 13. enrich_annotations
- 14. resolve_general_vs_specific
- 15. resolve_overlaps_with_priority
- 16. annotate_ace
- 17. annotate_notes_df
- 18. build_ace_matchers



In [8]:
# Functions to extract spans to the L or to the R of a central word.
def cut_left_at_break_with_offset(text, break_re, base_offset):
    """
    Cut left context at the last break marker and preserve absolute offset.

    Parameters
    ----------
    text : str
        Context fragment extracted from doc.text.
    break_re : compiled regex
        Regex defining break points.
    base_offset : int
        Absolute start offset of `text` in the original document.

    Returns
    -------
    tuple:
        cropped_text : str
        cropped_start_offset : int
    """
    matches = list(break_re.finditer(text))

    if matches:
        cut_pos = matches[-1].end()
        cropped_text = text[cut_pos:]
        cropped_start_offset = base_offset + cut_pos
        return cropped_text, cropped_start_offset

    return text, base_offset

def cut_right_at_break_with_offset(text, break_re, base_offset):
    """
    Cut right context at the first break marker and preserve absolute offset.

    Parameters
    ----------
    text : str
        Context fragment extracted from doc.text.
    break_re : compiled regex
        Regex defining break points.
    base_offset : int
        Absolute start offset of `text` in the original document.

    Returns
    -------
    tuple:
        cropped_text : str
        cropped_start_offset : int
    """
    m = break_re.search(text)

    if m:
        cropped_text = text[:m.start()]
        return cropped_text, base_offset

    return text, base_offset

In [9]:
# Need to apply a different configuration for perpetrators to the right.
def cut_right_for_perpetrator_with_offset(text, base_offset):
    """
    Cut right perpetrator context at the first perpetrator break marker
    and preserve absolute offset.

    Parameters
    ----------
    text : str
        Context fragment extracted from doc.text.
    base_offset : int
        Absolute start offset of `text` in the original document.

    Returns
    -------
    tuple:
        cropped_text : str
        cropped_start_offset : int
    """
    m = PERPETRATOR_BREAKS.search(text)

    if m:
        cropped_text = text[:m.start()]
        return cropped_text, base_offset

    return text, base_offset

In [10]:
# Function that takes out accents, maintains "ñ"
def strip_accents_keep_enye(text):
    result = []
    for char in text:
        if char in ("ñ", "Ñ"):
            result.append(char)
        else:
            decomposed = unicodedata.normalize("NFD", char)
            base = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
            result.append(base)
    return "".join(result)

In [11]:
# Introduced to be able to transform for ex: "violencia paterna" into: entity-> "maltrato", perpetrator-> "padre"
def normalize_perpetrator(term):
    term = term.lower().strip()
    if term in {"paterno", "paterna"}:
        return "padre"
    if term in {"materno", "materna"}:
        return "madre"
    return term

In [50]:
# Function to detect negation spans in a sentence that contains ACE terms.
def negation_flag_local(span, left_window=30, right_window=15):
    text = span.doc.text

    # Restrict to the same sentence
    L = max(span.start_char - left_window, span.sent.start_char)
    R = min(span.end_char + right_window, span.sent.end_char)

    raw_left = text[L:span.start_char]
    raw_right = text[span.end_char:R]

    # Keep only nearest local fragment, preserving absolute offsets
    ctx_left, ctx_left_start = cut_left_at_break_with_offset(
        raw_left, GENERAL_CLAUSE_BREAKS, L
    )
    ctx_right, ctx_right_start = cut_right_at_break_with_offset(
        raw_right, GENERAL_CLAUSE_BREAKS, span.end_char
    )

    # General negation cues
    m_left = NEG_CUES.search(ctx_left)
    if m_left:
        return (
            True,
            m_left.group(0),
            ctx_left_start + m_left.start(),
            ctx_left_start + m_left.end(),
        )

    m_right = NEG_CUES.search(ctx_right)
    if m_right:
        return (
            True,
            m_right.group(0),
            ctx_right_start + m_right.start(),
            ctx_right_start + m_right.end(),
        )
    
    """print("SPAN:", repr(span.text))
    print("raw_left:", repr(raw_left))
    print("ctx_left:", repr(ctx_left))
    print("raw_right:", repr(raw_right))
    print("ctx_right:", repr(ctx_right))
    print("NEG_CUES left:", NEG_CUES.search(ctx_left))
    print("NEG_CUES right:", NEG_CUES.search(ctx_right))
    print("---")"""

    # Special case: bare "no" immediately before the ACE mention
    # Examples: "no penetración", "pero no abuso", "sin embargo no maltrato"
    no_match = re.search(r"\bno\s*$", ctx_left, re.I)
    if no_match:
        return (
            True,
            no_match.group(0).strip(),
            ctx_left_start + no_match.start(),
            ctx_left_start + no_match.end(),
        )

    return (
        False,
        None,
        None,
        None,
    )

In [13]:
# Funtion to extract attribute values related to temporal cues, in childhood and recent times.
# Windows can be configured to the L and to the R.
# Modified its first version to include: past time cues, duration and frequency
# Modified its second version to include adult cues.
# Modified in its third version to separate frequency cues to another function (as frequency requiers larger windows)
def temporal_flags_local(span, left_window=40, right_window=40):
    text = span.doc.text

    # Restrict window to same sentence
    L = max(span.start_char - left_window, span.sent.start_char)
    R = min(span.end_char + right_window, span.sent.end_char)

    raw_left = text[L:span.start_char]
    raw_right = text[span.end_char:R]

    # Keep only nearest local fragment, preserving absolute offsets
    ctx_left, ctx_left_start = cut_left_at_break_with_offset(
        raw_left, TEMPORAL_BREAKS, L
    )
    ctx_right, ctx_right_start = cut_right_at_break_with_offset(
        raw_right, TEMPORAL_BREAKS, span.end_char
    )

    def search_with_offsets(pattern, left_text, left_start, right_text, right_start):
        """
        Search first in left context, then in right context.
        Return:
            cue_bool, cue_term, cue_start_char, cue_end_char
        """
        m_left = pattern.search(left_text)
        if m_left:
            return (
                True,
                m_left.group(0),
                left_start + m_left.start(),
                left_start + m_left.end(),
            )

        m_right = pattern.search(right_text)
        if m_right:
            return (
                True,
                m_right.group(0),
                right_start + m_right.start(),
                right_start + m_right.end(),
            )

        return False, None, None, None

    childhood_cue, childhood_term, childhood_start_char, childhood_end_char = search_with_offsets(
        CHILD_CUES, ctx_left, ctx_left_start, ctx_right, ctx_right_start
    )

    adult_cue, adult_term, adult_start_char, adult_end_char = search_with_offsets(
        ADULT_CUES, ctx_left, ctx_left_start, ctx_right, ctx_right_start
    )

    past_t_cue, past_t_term, past_t_start_char, past_t_end_char = search_with_offsets(
        PAST_TIME_CUES, ctx_left, ctx_left_start, ctx_right, ctx_right_start
    )

    recent_cue, recent_term, recent_start_char, recent_end_char = search_with_offsets(
        RECENT_CUES, ctx_left, ctx_left_start, ctx_right, ctx_right_start
    )

    current_cue, current_term, current_start_char, current_end_char = search_with_offsets(
        CURRENT_CUES, ctx_left, ctx_left_start, ctx_right, ctx_right_start
    )

    ongoing_cue, ongoing_term, ongoing_start_char, ongoing_end_char = search_with_offsets(
        ONGOING_CUES, ctx_left, ctx_left_start, ctx_right, ctx_right_start
    )

    # Avoid overlap: "desde hace 5 años" should not also give "hace 5 años"
    if ongoing_cue and past_t_cue:
        if past_t_term.lower() in ongoing_term.lower():
            past_t_cue = False
            past_t_term = None
            past_t_start_char = None
            past_t_end_char = None

    return {
        "childhood_cue": childhood_cue,
        "childhood_term": childhood_term,
        "childhood_start_char": childhood_start_char,
        "childhood_end_char": childhood_end_char,

        "adult_cue": adult_cue,
        "adult_term": adult_term,
        "adult_start_char": adult_start_char,
        "adult_end_char": adult_end_char,

        "past_t_cue": past_t_cue,
        "past_t_term": past_t_term,
        "past_t_start_char": past_t_start_char,
        "past_t_end_char": past_t_end_char,

        "recent_cue": recent_cue,
        "recent_term": recent_term,
        "recent_start_char": recent_start_char,
        "recent_end_char": recent_end_char,

        "current_cue": current_cue,
        "current_term": current_term,
        "current_start_char": current_start_char,
        "current_end_char": current_end_char,

        "ongoing_cue": ongoing_cue,
        "ongoing_term": ongoing_term,
        "ongoing_start_char": ongoing_start_char,
        "ongoing_end_char": ongoing_end_char,
    }

In [14]:
# New function to extract frequency cues, with larger windows.
def frequency_flags_local(span, left_window=40, right_window=80):
    text = span.doc.text

    # Restrict window to same sentence
    L = max(span.start_char - left_window, span.sent.start_char)
    R = min(span.end_char + right_window, span.sent.end_char)

    raw_left = text[L:span.start_char]
    raw_right = text[span.end_char:R]

    # Keep only nearest local fragment, preserving absolute offsets
    ctx_left, ctx_left_start = cut_left_at_break_with_offset(
        raw_left, TEMPORAL_BREAKS, L
    )
    ctx_right, ctx_right_start = cut_right_at_break_with_offset(
        raw_right, TEMPORAL_BREAKS, span.end_char
    )

    m_left = FREQUENCY_CUES.search(ctx_left)
    if m_left:
        return (
            True,
            m_left.group(0),
            ctx_left_start + m_left.start(),
            ctx_left_start + m_left.end(),
        )

    m_right = FREQUENCY_CUES.search(ctx_right)
    if m_right:
        return (
            True,
            m_right.group(0),
            ctx_right_start + m_right.start(),
            ctx_right_start + m_right.end(),
        )

    return (
        False,
        None,
        None,
        None,
    )

In [15]:
# Funtion to extract attribute values related to perpetrator cues.
# Windows can be configured to the L and to the R.
def perpetrator_flags_local(span, left_window=80, right_window=120, post_link_window=40):
    text = span.doc.text

    L = max(span.start_char - left_window, span.sent.start_char)
    R = min(span.end_char + right_window, span.sent.end_char)

    raw_left = text[L:span.start_char]
    raw_right = text[span.end_char:R]

    # Keep only nearest local fragment, preserving absolute offsets
    ctx_left, ctx_left_start = cut_left_at_break_with_offset(
        raw_left, GENERAL_CLAUSE_BREAKS, L
    )
    ctx_right, ctx_right_start = cut_right_for_perpetrator_with_offset(
        raw_right, span.end_char
    )

    # If adjectives "paterno/a" or "materno/a" appear with an ACE term, add as perpetrator padre/madre
    adj_fragment = ctx_right[:25]
    adj_match = PARENTAL_ADJ_CUES.search(adj_fragment)
    if adj_match:
        return (
            True,
            normalize_perpetrator(adj_match.group(1)),
            ctx_right_start + adj_match.start(1),
            ctx_right_start + adj_match.end(1),
        )

    # LEFT
    left_link = PERPETRATOR_LINK_CUES.search(ctx_left)
    if left_link:
        left_after = ctx_left[left_link.end():left_link.end() + post_link_window]
        left_after_start = ctx_left_start + left_link.end()

        left_term = PERPETRATOR_CUES.search(left_after)
        if left_term:
            return (
                True,
                normalize_perpetrator(left_term.group(0)),
                left_after_start + left_term.start(),
                left_after_start + left_term.end(),
            )

    # RIGHT
    right_link = PERPETRATOR_LINK_CUES.search(ctx_right)
    if right_link:
        right_after = ctx_right[right_link.end():right_link.end() + post_link_window]
        right_after_start = ctx_right_start + right_link.end()

        right_term = PERPETRATOR_CUES.search(right_after)
        if right_term:
            return (
                True,
                normalize_perpetrator(right_term.group(0)),
                right_after_start + right_term.start(),
                right_after_start + right_term.end(),
            )

    return (
        False,
        None,
        None,
        None,
    )

In [16]:
def discard_ambiguous_ace_match(doc, ann, window=50):
    text = doc.text

    start = ann["ace_start_char"]
    end = ann["ace_end_char"]
    span_text = ann["span"].lower()

    L = max(0, start - window)
    R = min(len(text), end + window)
    ctx = text[L:R].lower()

    # Filter: "abandono" in substance-use context
    if span_text == "abandono" and ABANDONO_SUBSTANCE_CONTEXT.search(ctx):
        return True

    return False

In [17]:
# Extract from records the terms that are in ACE term dictionary, via the matcher.
def extract_ace_terms(text: str, nlp, phrase_matcher, token_matcher):
    doc = nlp(text)

    ann = []

    # PhraseMatcher matches
    phrase_matches = phrase_matcher(doc)
    for match_id, start, end in phrase_matches:
        span = doc[start:end]
        ann.append({
            "concept_id": nlp.vocab.strings[match_id],
            "span": span.text,
            "ace_start_char": span.start_char,
            "ace_end_char": span.end_char,
        })

    # Token Matcher matches
    token_matches = token_matcher(doc)
    for match_id, start, end in token_matches:
        span = doc[start:end]
        ann.append({
            "concept_id": nlp.vocab.strings[match_id],
            "span": span.text,
            "ace_start_char": span.start_char,
            "ace_end_char": span.end_char,
        })

    # Deduplicate exact spans
    ann = sorted(ann, key=lambda x: (x["ace_start_char"], -x["ace_end_char"]))
    out, seen = [], set()
    for a in ann:
        key = (a["concept_id"], a["ace_start_char"], a["ace_end_char"])
        if key not in seen:
            out.append(a)
            seen.add(key)

    return doc, out

In [18]:
# Need to post.process matches to take into account cases as "maltrato físico y psicológico" -> need to extract
# "maltrato físico" and "maltrato psicológico"
# Generalized to include also "abuso" as head.
def add_coordination_derived_annotations(
    text: str,
    annotations: list[dict]
) -> tuple[list[dict], bool, int]:
    """
    Post-process coordinated expressions with shared head, such as:
    - 'maltrato físico y psicológico'
    - 'abuso físico y emocional'

    It creates a derived second annotation aligned to the original surface span
    (e.g. 'psicológico'), while preserving the shared head in metadata.

    Returns:
    - updated annotation list
    - coord_postprocessed: whether the rule fired at least once
    - n_coord_added: number of new annotations added by this rule
    """
    out = list(annotations)
    coord_postprocessed = False
    n_coord_added = 0

    text_lower = text.lower()

    for m in COORD_CUES.finditer(text):
        head = m.group(1)          # e.g. "maltrato" or "abuso"
        modifier = m.group(3)      # second coordinated adjective

        head_norm = strip_accents_keep_enye(head).lower()
        modifier_norm = strip_accents_keep_enye(modifier).lower()

        concept_id = COORD_TYPE_TO_CONCEPT.get(modifier_norm)
        if concept_id is None:
            continue

        start_char = text_lower.find(modifier.lower(), m.start(), m.end())
        if start_char == -1:
            continue

        end_char = start_char + len(modifier)
        surface_span = text[start_char:end_char]

        duplicate = any(
            a["concept_id"] == concept_id
            and a["ace_start_char"] == start_char
            and a["ace_end_char"] == end_char
            for a in out
        )
        if duplicate:
            continue

        out.append({
            "concept_id": concept_id,
            "span": surface_span,                            # real surface span in text
            "ace_start_char": start_char,
            "ace_end_char": end_char,
            "expanded_head": head,
            "normalized_span": f"{head} {surface_span}",    # semantic reconstruction
            "inferred_from_coordination": True,
        })

        coord_postprocessed = True
        n_coord_added += 1

    return out, coord_postprocessed, n_coord_added

In [19]:
# Introduce attributes that identify temporal, frequency, negation and perpetrator cues,
# now including cue offsets for later BRAT conversion.
def enrich_annotation(doc, ann: dict) -> dict:
    span = doc.char_span(ann["ace_start_char"], ann["ace_end_char"], alignment_mode="expand")

    if span is None:
        enriched = dict(ann)
        enriched.update({
            "negated": False,
            "negation_term": None,
            "negation_start_char": None,
            "negation_end_char": None,

            "childhood_cue": False,
            "childhood_term": None,
            "childhood_start_char": None,
            "childhood_end_char": None,

            "adult_cue": False,
            "adult_term": None,
            "adult_start_char": None,
            "adult_end_char": None,

            "past_t_cue": False,
            "past_t_term": None,
            "past_t_start_char": None,
            "past_t_end_char": None,

            "recent_cue": False,
            "recent_term": None,
            "recent_start_char": None,
            "recent_end_char": None,

            "current_cue": False,
            "current_term": None,
            "current_start_char": None,
            "current_end_char": None,

            "ongoing_cue": False,
            "ongoing_term": None,
            "ongoing_start_char": None,
            "ongoing_end_char": None,

            "frequency_cue": False,
            "frequency_term": None,
            "frequency_start_char": None,
            "frequency_end_char": None,

            "perpetrator_cue": False,
            "perpetrator_term": None,
            "perpetrator_start_char": None,
            "perpetrator_end_char": None,
        })
        return enriched

    negated, negation_term, negation_start_char, negation_end_char = negation_flag_local(
        span, left_window=30, right_window=15
    )

    temporal_info = temporal_flags_local(span, left_window=40, right_window=40)

    (
        frequency_cue,
        frequency_term,
        frequency_start_char,
        frequency_end_char,
    ) = frequency_flags_local(span, left_window=40, right_window=80)

    (
        perpetrator_cue,
        perpetrator_term,
        perpetrator_start_char,
        perpetrator_end_char,
    ) = perpetrator_flags_local(span, left_window=50, right_window=80)

    enriched = dict(ann)
    enriched.update({
        "negated": negated,
        "negation_term": negation_term,
        "negation_start_char": negation_start_char,
        "negation_end_char": negation_end_char,

        "childhood_cue": temporal_info["childhood_cue"],
        "childhood_term": temporal_info["childhood_term"],
        "childhood_start_char": temporal_info["childhood_start_char"],
        "childhood_end_char": temporal_info["childhood_end_char"],

        "adult_cue": temporal_info["adult_cue"],
        "adult_term": temporal_info["adult_term"],
        "adult_start_char": temporal_info["adult_start_char"],
        "adult_end_char": temporal_info["adult_end_char"],

        "past_t_cue": temporal_info["past_t_cue"],
        "past_t_term": temporal_info["past_t_term"],
        "past_t_start_char": temporal_info["past_t_start_char"],
        "past_t_end_char": temporal_info["past_t_end_char"],

        "recent_cue": temporal_info["recent_cue"],
        "recent_term": temporal_info["recent_term"],
        "recent_start_char": temporal_info["recent_start_char"],
        "recent_end_char": temporal_info["recent_end_char"],

        "current_cue": temporal_info["current_cue"],
        "current_term": temporal_info["current_term"],
        "current_start_char": temporal_info["current_start_char"],
        "current_end_char": temporal_info["current_end_char"],

        "ongoing_cue": temporal_info["ongoing_cue"],
        "ongoing_term": temporal_info["ongoing_term"],
        "ongoing_start_char": temporal_info["ongoing_start_char"],
        "ongoing_end_char": temporal_info["ongoing_end_char"],

        "frequency_cue": frequency_cue,
        "frequency_term": frequency_term,
        "frequency_start_char": frequency_start_char,
        "frequency_end_char": frequency_end_char,

        "perpetrator_cue": perpetrator_cue,
        "perpetrator_term": perpetrator_term,
        "perpetrator_start_char": perpetrator_start_char,
        "perpetrator_end_char": perpetrator_end_char,
    })

    return enriched

In [20]:
# Need to postprocess the cases where there are two matches, one more generic, example: "maltrato físico" and
# "maltrato". If there is an specific annotation, do not consider the general one.
def resolve_general_vs_specific(annotations: list[dict]) -> list[dict]:
    kept = []

    for ann in annotations:
        if ann["concept_id"] == "ACE_GENERAL_TRAUMA":
            contained_in_specific = any(
                other["concept_id"] in SPECIFIC_ACE_CONCEPTS
                and other["ace_start_char"] <= ann["ace_start_char"]
                and other["ace_end_char"] >= ann["ace_end_char"]
                for other in annotations
                if other is not ann
            )
            if contained_in_specific:
                continue

        kept.append(ann)

    return kept

In [21]:
def resolve_overlaps_with_priority(annotations):
    anns = sorted(
        annotations,
        key=lambda x: (
            x["ace_start_char"],
            -CONCEPT_PRIORITY.get(x["concept_id"], 0),
            -(x["ace_end_char"] - x["ace_start_char"])
        )
    )

    kept = []

    for ann in anns:
        overlap = any(
            not (ann["ace_end_char"] <= prev["ace_start_char"] or ann["ace_start_char"] >= prev["ace_end_char"])
            for prev in kept
        )
        if not overlap:
            kept.append(ann)

    return kept

In [22]:
# Calls the different functions to perform the extraction of spans and the postprocessing
def annotate_ace(text: str, nlp, phrase_matcher, token_matcher):
    """
    Run the ACE annotation pipeline on one text.

    Returns:
    - out: final list of annotation dicts
    - coord_postprocessed: whether the coordination post-processing rule fired
    - n_coord_added: number of annotations initially added by that rule
    - n_final_coord_annotations: number of coordination-derived annotations
      that survived all filtering/resolution steps
    """
    # 1) Extract spans
    doc, annotations = extract_ace_terms(text, nlp, phrase_matcher, token_matcher)

    # 2) Augment spans with coordination-based inferred annotations
    annotations, coord_postprocessed, n_coord_added = add_coordination_derived_annotations(
        text, annotations
    )

    # 3) Filter ambiguous spans before enrichment
    annotations = [
        ann for ann in annotations
        if not discard_ambiguous_ace_match(doc, ann)
    ]

    # 4) Enrich spans
    annotations = [enrich_annotation(doc, ann) for ann in annotations]

    # 5) Resolve general vs specific overlaps
    annotations = resolve_general_vs_specific(annotations)

    # 6) Resolve priority-based overlaps
    annotations = resolve_overlaps_with_priority(annotations)

    # 7) Final ordering
    annotations = sorted(
        annotations,
        key=lambda x: (x["ace_start_char"], x["ace_end_char"], x["concept_id"])
    )

    # 8) Final deduplication
    out = []
    seen = set()

    for a in annotations:
        key = (a["concept_id"], a["ace_start_char"], a["ace_end_char"])
        if key not in seen:
            out.append(a)
            seen.add(key)

    # 9) Count coordination-derived annotations that survived
    n_final_coord_annotations = sum(
        a.get("inferred_from_coordination", False) for a in out
    )

    return out, coord_postprocessed, n_coord_added, n_final_coord_annotations

In [23]:
# Function to apply annotation over a given dataframe, typically loaded from an excel file (or any alternative)
def annotate_notes_df(
    df_in,
    note_id_column_orig,
    text_column_orig,
    cols_order,
    offset_cols,
    nlp,
    phrase_matcher,
    token_matcher,
    note_id_column_dest="ehr_id",
    text_column_dest="ehr_history"
):
    all_ann = []
    notes_audit = []

    for _, row in df_in.iterrows():
        note_text = row[text_column_orig]
        note_id = row[note_id_column_orig]

        if pd.isna(note_text) or not str(note_text).strip():
            notes_audit.append({
                note_id_column_dest: note_id,
                "coord_postprocessed": False,
                "n_coord_added": 0,
                "n_final_coord_annotations": 0,
                "has_any_annotation": False,
            })
            continue

        ann, coord_postprocessed, n_coord_added, n_final_coord_annotations = annotate_ace(
            str(note_text), nlp, phrase_matcher, token_matcher
        )

        notes_audit.append({
            note_id_column_dest: note_id,
            "coord_postprocessed": coord_postprocessed,
            "n_coord_added": n_coord_added,
            "n_final_coord_annotations": n_final_coord_annotations,
            "has_any_annotation": bool(ann),
        })

        if not ann:
            continue

        ann_df = pd.DataFrame(ann)
        ann_df[note_id_column_dest] = note_id
        ann_df[text_column_dest] = note_text
        ann_df["coord_postprocessed"] = coord_postprocessed
        ann_df["n_coord_added"] = n_coord_added
        ann_df["n_final_coord_annotations"] = n_final_coord_annotations

        if "inferred_from_coordination" not in ann_df.columns:
            ann_df["inferred_from_coordination"] = pd.Series(
                False, index=ann_df.index, dtype="boolean"
            )
        else:
            ann_df["inferred_from_coordination"] = (
                ann_df["inferred_from_coordination"]
                .astype("boolean")
                .fillna(False)
            )

        # Ensure all columns exist before concat (avoids dtype ambiguity)
        for col in cols_order:
            if col not in ann_df.columns:
                ann_df[col] = pd.NA

        # Enforce column order
        ann_df = ann_df[cols_order]

        all_ann.append(ann_df)

    if all_ann:
        annotations_df = pd.concat(all_ann, ignore_index=True)

        for col in offset_cols:
            if col in annotations_df.columns:
                annotations_df[col] = annotations_df[col].astype("Int64")

        bool_cols = ["inferred_from_coordination", "coord_postprocessed"]
        for col in bool_cols:
            if col in annotations_df.columns:
                annotations_df[col] = annotations_df[col].astype("boolean")

        int_cols = ["n_coord_added", "n_final_coord_annotations"]
        for col in int_cols:
            if col in annotations_df.columns:
                annotations_df[col] = annotations_df[col].astype("Int64")

    else:
        annotations_df = pd.DataFrame(columns=cols_order)

    notes_audit_df = pd.DataFrame(notes_audit)

    return annotations_df, notes_audit_df

In [24]:
# Function that builds the Spacy matcher and adds: lower letters and takes out accents, maintains "ñ"
# Function that builds:
# - PhraseMatcher for ACE dictionary terms
# - Matcher for token-level patterns
# Lowercase matching is used in both.
def build_ace_matchers(nlp, ace_terms):
    phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    token_matcher = Matcher(nlp.vocab)

    # -------------------------
    # PhraseMatcher from dictionary
    # -------------------------
    for concept_id, terms in ace_terms.items():
        pattern_texts = set()

        if not isinstance(terms, list):
            continue

        for term in terms:
            if not isinstance(term, str):
                continue

            term = term.strip()
            if not term:
                continue

            pattern_texts.add(term.lower())
            pattern_texts.add(strip_accents_keep_enye(term).lower())

        if pattern_texts:
            patterns = [nlp.make_doc(t) for t in pattern_texts]
            phrase_matcher.add(concept_id, patterns)

    # -------------------------
    # Token-level ACE patterns
    # -------------------------

    # abusado/abusada sexualmente
    token_matcher.add(
        "ACE_SEXUAL_ABUSE",
        [[
            {"LOWER": {"IN": ["abusado", "abusada"]}},
            {"LOWER": "sexualmente"}
        ]]
    )

    # agredido/agredida sexualmente
    token_matcher.add(
        "ACE_SEXUAL_ABUSE",
        [[
            {"LOWER": {"IN": ["agredido", "agredida"]}},
            {"LOWER": "sexualmente"}
        ]]
    )

    # víctima de abuso sexual / victima de abuso sexual
    token_matcher.add(
        "ACE_SEXUAL_ABUSE",
        [[
            {"LOWER": {"IN": ["víctima", "victima"]}},
            {"LOWER": "de"},
            {"LOWER": "abuso"},
            {"LOWER": "sexual"}
        ]]
    )

    return phrase_matcher, token_matcher

##### Normalization functions

In [25]:
# NORMALIZATION FUNCTIONS

In [26]:
# Negation
def normalize_negation(negated):
    """
    negation: {true, false}
    """
    return "true" if negated else "false"

In [27]:
# Childhood and adult cues
def normalize_childhood(childhood_cue, adult_cue):
    """
    childhood: {true, false, unspecified}
    """
    if childhood_cue and not adult_cue:
        return "true"
    if adult_cue and not childhood_cue:
        return "false"
    return "unspecified"


def normalize_adult(adult_cue, childhood_cue):
    """
    adult: {true, false, unspecified}
    """
    if adult_cue and not childhood_cue:
        return "true"
    if childhood_cue and not adult_cue:
        return "false"
    return "unspecified"

In [28]:
# Temporality cues
def normalize_temporality(
    past_t_cue,
    recent_cue,
    current_cue,
    ongoing_cue,
    childhood_cue=False
):
    """
    temporality: {past, recent, current, ongoing, unspecified}

    Priority:
    ongoing > current > recent > past > inferred past from childhood > unspecified
    """

    if ongoing_cue:
        return "ongoing"

    if current_cue:
        return "current"

    if recent_cue:
        return "recent"

    if past_t_cue:
        return "past"

    # Inference rule:
    # childhood implies past if no stronger temporal cue exists
    if childhood_cue:
        return "past"

    return "unspecified"

In [29]:
# Frequency normalization function:
def normalize_frequency(frequency_cue, frequency_term):
    """
    frequency: {single, repeated, recurrent, unspecified}

    Note:
    - 'ocasionalmente' / 'ocasional' are currently mapped to 'repeated'
      following your current provisional decision.
    """
    if not frequency_cue or not frequency_term:
        return "unspecified"

    term = frequency_term.strip().lower()

    single_terms = {
        "en una ocasion",
        "en una ocasión",
        "en una sola ocasion",
        "en una sola ocasión",
        "en una unica ocasion",
        "en una única ocasión",
        "una vez",
        "solo una vez",
        "solamente una vez",
    }

    recurrent_terms = {
        "de forma recurrente",
        "de manera recurrente",
        "recurrentemente",
    }

    repeated_terms = {
        "en varias ocasiones",
        "en múltiples ocasiones",
        "en multiples ocasiones",
        "en repetidas ocasiones",
        "en repetidas oportunidades",
        "en repetida oportunidad",
        "en más de una ocasión",
        "en mas de una ocasion",
        "varias veces",
        "muchas veces",
        "más de una vez",
        "mas de una vez",
        "repetidamente",
        "de forma repetida",
        "de manera repetida",
        "ocasionalmente",
        "ocasional",
    }

    if term in single_terms:
        return "single"
    if term in recurrent_terms:
        return "recurrent"
    if term in repeated_terms:
        return "repeated"

    return "unspecified"


In [30]:
# Perpetrator normalization function:
def normalize_perpetrator_category(perpetrator_cue, perpetrator_term):
    """
    perpetrator: {family_member, partner, known_other, unknown_other, unspecified}

    Assumes perpetrator_term is the surface term returned by the cue layer.
    If needed, normalize_perpetrator(term) can be applied before this function.
    """
    if not perpetrator_cue or not perpetrator_term:
        return "unspecified"

    term = perpetrator_term.strip().lower()

    family_terms = {
        "padre",
        "madre",
        "hermano",
        "hermana",
        "tio",
        "tía",
        "tío",
        "tia",
        "abuelo",
        "abuela",
        "primo",
        "prima",
        "padre de su hijo",
        "madre de su hijo",
        "marido de tia",
        "marido de tía",
    }

    partner_terms = {
        "pareja",
        "ex",
        "ex pareja",
        "novio",
        "novia",
        "marido",
        "mujer",
        "esposo",
        "esposa",
    }

    known_other_terms = {
        "amigo",
        "amiga",
        "vecino",
        "vecina",
        "compañero",
        "compañera",
        "companero",
        "companera",
        "conocido",
        "conocida",
        "profesor",
        "profesora",
        "maestro",
        "maestra",
    }

    unknown_other_terms = {
        "desconocido",
        "desconocida",
    }

    if term in family_terms:
        return "family_member"
    if term in partner_terms:
        return "partner"
    if term in known_other_terms:
        return "known_other"
    if term in unknown_other_terms:
        return "unknown_other"

    return "unspecified"

In [31]:
# Wrapper functions to apply normalization over the annotation dataframe
# Per record
def normalize_annotation_record(ann):
    out = dict(ann)

    out["negation"] = normalize_negation(out.get("negated", False))

    out["childhood"] = normalize_childhood(
        out.get("childhood_cue", False),
        out.get("adult_cue", False),
    )

    out["adult"] = normalize_adult(
        out.get("adult_cue", False),
        out.get("childhood_cue", False),
    )

    out["temporality"] = normalize_temporality(
    out.get("past_t_cue", False),
    out.get("recent_cue", False),
    out.get("current_cue", False),
    out.get("ongoing_cue", False),
    out.get("childhood_cue", False),
)

    out["frequency"] = normalize_frequency(
        out.get("frequency_cue", False),
        out.get("frequency_term", None),
    )

    out["perpetrator"] = normalize_perpetrator_category(
        out.get("perpetrator_cue", False),
        out.get("perpetrator_term", None),
    )

    return out

# Per dataframe
def normalize_annotations_df(annotations_df, normalized_cols_order, debug=False):
    normalized_records = annotations_df.apply(
        lambda row: normalize_annotation_record(row.to_dict()),
        axis=1
    ).tolist()

    # print("First normalized record:")
    # print(normalized_records[0])

    normalized_df = pd.DataFrame(normalized_records)

    # print("Columns before reindex:")
    # print(normalized_df.columns.tolist())

    if debug:
        print(normalized_df.head())

    normalized_df = normalized_df.reindex(columns=normalized_cols_order)
    return normalized_df

In [32]:
# Wrapper function to apply the annotation and the normalization over a single sentence, for testing purposes
def annotate_and_normalize_ace(text, nlp, phrase_matcher, token_matcher):
    """
    Run full ACE pipeline on one sentence/text:
    1. regex-based ACE annotation
    2. normalization of each annotation record

    Returns:
        list of normalized annotation dictionaries
    """
    annotations, coord_postprocessed, n_coord_added, n_final_coord_annotations = annotate_ace(
        text, nlp, phrase_matcher, token_matcher
    )

    normalized_annotations = [
        normalize_annotation_record(a)
        for a in annotations
    ]

    return {
        "annotations": normalized_annotations,
        "coord_postprocessed": coord_postprocessed,
        "n_coord_added": n_coord_added,
        "n_final_coord_annotations": n_final_coord_annotations,
    } 

### Load abusos file

In [33]:
# Recover the path
directory_path = os.getcwd()
# root_path = os.path.dirname(directory_path)
print(directory_path)
# print(root_path)

c:\Users\adeli\Documents 4-Q2\Practicas-ACE\anotacionACE\ACE Project


In [34]:
# Patients in the file adversidad_precoz.xlsx (confirmed cases)
ace_cases_filepath = os.path.join(directory_path, 'data\\', "ace_confirmed_cases_eng.xlsx")
print(ace_cases_filepath)
ace_cases_df = pd.read_excel(ace_cases_filepath, na_values=["#NULL!"])

c:\Users\adeli\Documents 4-Q2\Practicas-ACE\anotacionACE\ACE Project\data\ace_confirmed_cases_eng.xlsx


In [35]:
len(ace_cases_df)

89

In [36]:
ace_cases_df.head()

,Anonymized_Temporal,RC,Reason_for_Consultation,Episode,Episode_Date,Sex,DoB,Assessment_Date,Psychiatric_History,Medical_History,Subjective_Impressions,Grade,Recent_Suicidal_Planning,Grade_A,EJEI_Coded_Diagnostic_CIE,EJEI_Diagnostic,Medical_Damage_Severity_From_Current_Attempt,Family_History_of_Suicide_Attempts,EJEI_Personality_and_Behavior_Disorder_L,EJEI_Personality_and_Behavior_Disorder,Recent_Suicidal_Ideation,EJEI_Somatic_Disorder_Cod1,EJEI_Somatic_Disorder_Text,Number_of_Previous_Suicide_Attempts,EJEI_Psychosocial_and_Environmental_Issues,Discharge_Treatment,Psychopathological_Examination,Last_Episode,Num_Episodes,Abuse,Maltreatment,Filter
0,216535,IDEACIÓN AUTOLÍTICA,Transtornos de conducta,U2022,2022-06-20 13:10:00,M,1971-02-12,2022-06-21 09:49:17,"Primer contacto con psiquiatría en el 2008, cuando tuvo varios ingresos por intoxicaciones etilicas y abuso de alcohol que relaciona con haber sido victima de malos tratos por parte del padre de su hijo, y abusos sexuales en la infancia.\nRealizó desentoxicacion en el Centro Los Almendros, por un año. En el 2012 empezó seguimiento en San Martin de Valdeiglesias. \nEstuvo ingresada en UDA en Noviembre 2016, 1 mes.\nUn ingreso psiquiátrico en el HRJC en agosto de 2016.\n\nTóxicos:- Alcohol:consumo perjudicial sobre los 25 años, de tipo dipsomaniaco. Niega periodos de consumo diario. Incremento de consumo en el 2008, en contexto de maltratos. Luego abandonó y se ha mantenido abstinente. Refiere en la actualidad consumo ocasional.\n- Cocaína y cannabis: niega consumo a diario pero reconoce consumo varias veces por semana\n- No otros tóxicos.\n\nSB: Natural de España. Madre alcoholica, desde pequeña se quedaba con sus vecinos, que luego acabaron solicitando la tutela. Vivió con sus padres biológicos hasta los 7 años.\nVivía con su hijo, hasta que le retiraron tutela servicio sociales por indicios de negligencia en el cuidado, en los periodos en los que la paciente consumía. \nTiene pareja estable. Trabajaba cuidando a personas mayores.",NaN,NaN,NaN,No,No,NaN,Sindrome de dependencia a alcohol\nSindrome de dependencia a cannabis\nSindrome de dependencia a cocaina,NaN,No,NaN,NaN,No,NaN,NaN,1,NaN,7-3-19\n\n- Cymbalta 60mg 1-1-0\n- Rivotril gotas 5-0-5 \n- Risperidona 1mg 0-0-0-1\n- Antabús 1-0-0,"Orientada auto y alopsiquicamente. Angustia moderada. No alteraciones de la psicomotricidad ni de la conducta. Actitud histrioniforme. Afecto depresivo, ánimo bajo de larga data reactivo a situación vital. No alteraciones del curso o contenido del pensamiento. Discurso coherente y estructurado. Sueño y apetito conservado. No alteraciones sensoperceptivas. Refiere ideas de muerte, sin planificación ni estructuración. Fobias de impulsión presentes. Escaso apoyo sociofamiliar.",1,31,1,1,1
1,232165,VIDEOCONSULTA SUCESIVA PSQ,NaN,C2023,2023-04-24 13:24:00,M,1986-12-31,2023-04-24 13:25:23,"- Inicio de síntomas psicóticos (ideación persecutoria, alucinaciones auditivas y visuales), según refiere, a los 12 años aproximadamente. \n- 1ª atención psiquiátrica a los 14 años (Clínica la Zarzuela). Pautan tratamiento con risperidona que toma durante 2 meses aproximadamente. Tras sobreingesta por accidente doméstico abandona el tratamiento.\n- Vista en una ocasión a los 16 años en CSM Villalba (Dra. Sánchez) quien propone ingreso para observación y diagnóstico, pero la familia rechaza esta opción. \n- Consumo de drogas durante la adolescencia con criterios de abuso de cocaína y cannabis y dependencia a alcohol. Abstinencia de tóxicos desde hace mas de 10 años.\n- Inicia tratamiento a los 21 años con psiquiatra privado (Dr. Ramón Campos), quien diagnostica a la paciente de desdoblamiento de la personalidad con esquizotipia y pauta tratamiento con citalopram, risperidona, reboxetina y clonazepam.\n- Primer ingreso psiquiátrico en febrero 2009 en la Clínica San Juan de Dios con diagnóstico al alta de Esquizofrenia paranoide y tratamiento con reboxetina 8 mg/d, olanzapina 20 mg/d, levomepromazina 100 mg/d, tri

### Annotation pipeline

In [37]:
# Load the dictionary of ACE terms
with open("ace_terms_dictionary.json", "r", encoding="utf-8") as f:
    ace_terms = json.load(f)

In [38]:
# Load the spanish medium model, build the matcher.
nlp = spacy.load("es_core_news_md", disable=["ner", "parser", "tagger"])
nlp.add_pipe("sentencizer")
phrase_matcher, token_matcher = build_ace_matchers(nlp, ace_terms)

In [39]:
# Choose the id column in the ACE cases dataframe to use and the text column to annotate
case_history_col = "Psychiatric_History"
case_id_col = "Anonymized_Temporal"

In [40]:
# Choose the names of the id and history columns in the resulting dataframe
id_col_name= "ehr_id"
history_col_name = "ehr_history"


In [41]:
# Base non-offset columns in desired semantic order
base_cols = [
    id_col_name,
    "concept_id",
    "span",

    # Negation
    "negated",
    "negation_term",

    # Childhood
    "childhood_cue",
    "childhood_term",

    # Adult
    "adult_cue",
    "adult_term",

    # Frequency
    "frequency_cue",
    "frequency_term",

    # Past temporality
    "past_t_cue",
    "past_t_term",

    # Recent temporality
    "recent_cue",
    "recent_term",

    # Current temporality
    "current_cue",
    "current_term",

    # Ongoing temporality
    "ongoing_cue",
    "ongoing_term",

    # Perpetrator
    "perpetrator_cue",
    "perpetrator_term",

    # Original note text
    history_col_name,
]

# Offset columns defined once centrally
offset_cols = [
    "ace_start_char",
    "ace_end_char",
    "negation_start_char",
    "negation_end_char",
    "childhood_start_char",
    "childhood_end_char",
    "adult_start_char",
    "adult_end_char",
    "frequency_start_char",
    "frequency_end_char",
    "past_t_start_char",
    "past_t_end_char",
    "recent_start_char",
    "recent_end_char",
    "current_start_char",
    "current_end_char",
    "ongoing_start_char",
    "ongoing_end_char",
    "perpetrator_start_char",
    "perpetrator_end_char",
]

# Post-processing layer, allows to audit whether the coordination-based rule fired and how many annotations it added, 
# and also to keep track of how many of those survived the full pipeline.
coord_cols = [
    "expanded_head",
    "normalized_span",
    "inferred_from_coordination",
    "coord_postprocessed",
    "n_coord_added",
    "n_final_coord_annotations",
]

# Final ordered schema
cols_order = base_cols + offset_cols + coord_cols

In [42]:
annotations_df, notes_audit_df = annotate_notes_df(
    df_in=ace_cases_df,
    note_id_column_orig=case_id_col,
    text_column_orig=case_history_col,
    cols_order=cols_order,
    offset_cols=offset_cols,
    nlp=nlp,
    phrase_matcher=phrase_matcher,
    token_matcher=token_matcher,
    note_id_column_dest=id_col_name,
    text_column_dest=history_col_name
)

SPAN: 'victima'
raw_left: ' que relaciona con haber sido '
ctx_left: ' que relaciona con haber sido '
raw_right: ' de malos trato'
ctx_right: ' de malos trato'
NEG_CUES left: None
NEG_CUES right: None
---
SPAN: 'malos tratos'
raw_left: 'ona con haber sido victima de '
ctx_left: 'ona con haber sido victima de '
raw_right: ' por parte del '
ctx_right: ' por parte del '
NEG_CUES left: None
NEG_CUES right: None
---
SPAN: 'abusos sexuales'
raw_left: 'parte del padre de su hijo, y '
ctx_left: ' y '
raw_right: ' en la infancia'
ctx_right: ' en la infancia'
NEG_CUES left: None
NEG_CUES right: None
---
SPAN: 'abusos'
raw_left: 'parte del padre de su hijo, y '
ctx_left: ' y '
raw_right: ' sexuales en la'
ctx_right: ' sexuales en la'
NEG_CUES left: None
NEG_CUES right: None
---
SPAN: 'Madre alcoholica'
raw_left: ''
ctx_left: ''
raw_right: ', desde pequeña'
ctx_right: ''
NEG_CUES left: None
NEG_CUES right: None
---
SPAN: 'negligencia'
raw_left: 'icio sociales por indicios de '
ctx_left: 'icio soci

C:\Users\adeli\AppData\Local\Temp\ipykernel_24432\2878103088.py:75: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  annotations_df = pd.concat(all_ann, ignore_index=True)


In [43]:
aux = annotations_df.drop(columns=history_col_name).head(1)

In [44]:
print(aux.T)

                                             0
ehr_id                                  216535
concept_id                  ACE_GENERAL_TRAUMA
span                                   victima
negated                                  False
negation_term                             None
childhood_cue                            False
childhood_term                            None
adult_cue                                False
adult_term                                None
frequency_cue                            False
frequency_term                            None
past_t_cue                               False
past_t_term                               None
recent_cue                               False
recent_term                               None
current_cue                              False
current_term                              None
ongoing_cue                              False
ongoing_term                              None
perpetrator_cue                           True
perpetrator_t

In [45]:
annotations_df["ehr_id"].nunique()

89

In [46]:
annotations_df.columns

Index(['ehr_id', 'concept_id', 'span', 'negated', 'negation_term',
       'childhood_cue', 'childhood_term', 'adult_cue', 'adult_term',
       'frequency_cue', 'frequency_term', 'past_t_cue', 'past_t_term',
       'recent_cue', 'recent_term', 'current_cue', 'current_term',
       'ongoing_cue', 'ongoing_term', 'perpetrator_cue', 'perpetrator_term',
       'ehr_history', 'ace_start_char', 'ace_end_char', 'negation_start_char',
       'negation_end_char', 'childhood_start_char', 'childhood_end_char',
       'adult_start_char', 'adult_end_char', 'frequency_start_char',
       'frequency_end_char', 'past_t_start_char', 'past_t_end_char',
       'recent_start_char', 'recent_end_char', 'current_start_char',
       'current_end_char', 'ongoing_start_char', 'ongoing_end_char',
       'perpetrator_start_char', 'perpetrator_end_char', 'expanded_head',
       'normalized_span', 'inferred_from_coordination', 'coord_postprocessed',
       'n_coord_added', 'n_final_coord_annotations'],
      dtype='

In [47]:
current_file_name = "annotations_ace_confirmed_cases_v6.xlsx"
annotations_df.to_excel(current_file_name   , index=False)

In [48]:
# To recover the annotations file
annotations_df = pd.read_excel(current_file_name)

### Test sentences

In [84]:
annotations_df["ehr_id"].unique()

array([  216535,   232165,   262678,   269487,   271022,   305129,
         349669,   365985,   386053,   389030,   403205,   448577,
         464272,   495492,   504137,   564433,   595962,   649848,
         703682,   706171,   784004,   796589,   810687,   814726,
         916274,   923573,   941569,   950325,   955376,   961679,
         976166,   999543,  1025422,  1042052,  1055670,  1072754,
        1077142,  1086522,  1087796,  1113996,  1140733,  1156968,
        1171351,  1202384,  1206473,  1225585,  1227245,  1304145,
        1309521,  1311540,  1332221,  1346534,  1364855,  1369635,
        1375166,  1390426,  1391835,  1461127,  1469042,  1470079,
        1479292,  1492571,  1495588,  1508243,  1518800,  1559801,
        1858236,  2100345,  2173316,  2366090,  4041621,  4072292,
       15036828, 20093050, 20283167, 20354316, 20384566, 20387955,
       22083896, 22190123, 22190230, 22217146, 22301469, 22394845,
       23038503, 24021942, 24087626, 24111281, 25745955], dtyp

In [85]:
# Examine a given EHR 
ehr_id = 1055670
row = annotations_df.loc[annotations_df["ehr_id"] == ehr_id]
print(row["ehr_history"].iloc[0])

APS: No RAMc. No refiere otro tipo de alergias. Crisis epilépticas neonatales, siendo dado de alta por neurología en 2005. Estuvo en tratamiento con fenobarbital y luminal.

APP: No refiere. No antecedentes de abusos ni maltrato.
Tóxicos: Tabaco 1-3 cigarros/semana. Alcohol en fiestas. Niega consumo de otros tóxicos.

SB: Convive con sus padres. Madre, Mercedes de 45 años, trabaja en limpieza. Padre, Ricardo, de 51 años, jubilado por IL(albañil). Cuenta con un hermano de 21 años. Buena relación familiar.

Historia psicobiográfica:
Embarazo con DM gestacional. Parto inducido en la semana 37 por elevación de glucosa. Peso 2450. Estuvo en la Unidad neonatología 21 días. No ictericia neonatal. Crisis convulsiva a las 48h del nacimiento, en tto hasta los 5 años.
Lactancia materna: 1 mes.
Comedor: bueno.
Sueño: demasiado siempre.
Deambulación: 19 meses, con gateo previo.
Habla:24 meses. "Tardo por lesión hemisférica izquierda" (no saben precisar y no aporta informes).
Control de esfínteres: 

In [86]:
row.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
76,1055670,ACE_GENERAL_TRAUMA,abusos,True,No antecedentes de,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,210,216,191.0,209.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
77,1055670,ACE_GENERAL_TRAUMA,maltrato,True,No antecedentes de,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,220,228,191.0,209.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [87]:
note5  = "Inicio de consumo de alcohol a los 15 años y abandono a los 34 años"
annotate_ace(note5, nlp, phrase_matcher, token_matcher)
# Correct result, abandono not related to ACE

([], False, 0, 0)

In [88]:
note0="Episodio depresivo tras fallecimiento de hijo hace 20 años Refiere maltrato en la infancia"
annotate_ace(note0, nlp, phrase_matcher, token_matcher)


([{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'maltrato',
   'ace_start_char': 67,
   'ace_end_char': 75,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'en la infancia',
   'childhood_start_char': 76,
   'childhood_end_char': 90,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': None,
   '

In [89]:
note1="haber sido victima de malos tratos por parte del padre de su hijo, y abusos sexuales en la infancia."
annotate_ace(note1, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'victima',
   'ace_start_char': 11,
   'ace_end_char': 18,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': None,
   'perpetra

In [90]:
note="maltrato físico y psicológico en el seno familiar por parte de su madre y una hermana"
annotate_ace(note, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_PHYSICAL_ABUSE',
   'span': 'maltrato físico',
   'ace_start_char': 0,
   'ace_end_char': 15,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': None,
   'p

In [91]:
note2="Refiere maltrato por parte del padre de niña. Abusos sex por parte de marido de tia en una ocasion,..."
annotate_ace(note2, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'maltrato',
   'ace_start_char': 8,
   'ace_end_char': 16,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'de niña',
   'childhood_start_char': 37,
   'childhood_end_char': 44,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': None,
   'perpetra

In [92]:
note4 = "por amigo cuando tenía 15 años"
annotate_ace(note4, nlp, phrase_matcher, token_matcher)

([], False, 0, 0)

In [93]:
note3= "intento de abuso sexual por su abuelo a los 6 a. No tuvo mucha "
annotate_ace(note3, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_ATTEMPTED_ABUSE',
   'span': 'intento de abuso sexual',
   'ace_start_char': 0,
   'ace_end_char': 23,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'a los 6 a',
   'childhood_start_char': 38,
   'childhood_end_char': 47,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': 

In [94]:
note4="Intento de abuso sexual del padre con Carmen."
annotate_ace(note4, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_ATTEMPTED_ABUSE',
   'span': 'Intento de abuso sexual',
   'ace_start_char': 0,
   'ace_end_char': 23,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': No

In [95]:
note6 = "Los abusos ocurrieron cuando Joe tenía 6 años de edad"
annotate_ace(note6, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'abusos',
   'ace_start_char': 4,
   'ace_end_char': 10,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'cuando Joe tenía 6 años',
   'childhood_start_char': 22,
   'childhood_end_char': 45,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': None

In [96]:
note7= "Comenta que su hijo, cuando tenía 7 años de edad, fue abusado sexualmente por un vecino."
annotate_ace(note7, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_SEXUAL_ABUSE',
   'span': 'abusado sexualmente',
   'ace_start_char': 54,
   'ace_end_char': 73,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'cuando tenía 7 años de edad',
   'childhood_start_char': 21,
   'childhood_end_char': 48,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency

In [97]:
note8="hubo tocamientos, pero no penetración"
annotate_ace(note8, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_SEXUAL_ABUSE',
   'span': 'tocamientos',
   'ace_start_char': 5,
   'ace_end_char': 16,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': None,
   'perpetr

In [98]:
note9 = "No antecedentes de abusos ni maltrato."
annotate_ace(note9, nlp, phrase_matcher, token_matcher)

([{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'abusos',
   'ace_start_char': 19,
   'ace_end_char': 25,
   'negated': True,
   'negation_term': 'No antecedentes de',
   'negation_start_char': 0,
   'negation_end_char': 18,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': None,
   

##### Test concrete parts of what is annotated

In [99]:
annotations_df.columns

Index(['ehr_id', 'concept_id', 'span', 'negated', 'negation_term',
       'childhood_cue', 'childhood_term', 'adult_cue', 'adult_term',
       'frequency_cue', 'frequency_term', 'past_t_cue', 'past_t_term',
       'recent_cue', 'recent_term', 'current_cue', 'current_term',
       'ongoing_cue', 'ongoing_term', 'perpetrator_cue', 'perpetrator_term',
       'ehr_history', 'ace_start_char', 'ace_end_char', 'negation_start_char',
       'negation_end_char', 'childhood_start_char', 'childhood_end_char',
       'adult_start_char', 'adult_end_char', 'frequency_start_char',
       'frequency_end_char', 'past_t_start_char', 'past_t_end_char',
       'recent_start_char', 'recent_end_char', 'current_start_char',
       'current_end_char', 'ongoing_start_char', 'ongoing_end_char',
       'perpetrator_start_char', 'perpetrator_end_char', 'expanded_head',
       'normalized_span', 'inferred_from_coordination', 'coord_postprocessed',
       'n_coord_added', 'n_final_coord_annotations'],
      dtype='

In [100]:
# Look for what is recover in "past_t_cue" 
df_filtered = annotations_df[annotations_df["past_t_cue"] == True]
df_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
120,1346534,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,False,NaN,False,NaN,False,NaN,False,NaN,True,hace 20 años,False,NaN,False,NaN,False,NaN,False,NaN,1166,1182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1183.0,1195.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [101]:
# Look for what is recover in "recent_term"
df_filtered = annotations_df[annotations_df["recent_cue"] == True]
df_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations


In [102]:
# Look for what is recover in "current_cue" 
df_filtered = annotations_df[annotations_df["current_cue"] == True]
df_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
184,22190123,ACE_GENERAL_TRAUMA,maltrato,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,en la actualidad,False,NaN,False,NaN,2816,2824,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2841.0,2857.0,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
187,22190123,ACE_GENERAL_TRAUMA,violencia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,actualmente,False,NaN,False,NaN,5284,5293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5250.0,5261.0,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [103]:
# Look for what is recover in past_t_term"
df_filtered = annotations_df[annotations_df["past_t_cue"] == True]
df_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
120,1346534,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,False,NaN,False,NaN,False,NaN,False,NaN,True,hace 20 años,False,NaN,False,NaN,False,NaN,False,NaN,1166,1182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1183.0,1195.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [104]:
# Look for what is recover in "ongoing_cue"
df_filtered = annotations_df[annotations_df["ongoing_cue"] == True]
df_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
139,1492571,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,desde hace 3 años,False,NaN,243,259,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260.0,277.0,NaN,NaN,NaN,NaN,False,False,0,0


In [105]:
# Look what is recovered as frequency_term XXX
df_filtered = annotations_df[annotations_df["frequency_cue"] == True]
df_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
10,269487,ACE_SEXUAL_ABUSE,Abusos sex,False,NaN,False,NaN,False,NaN,True,en una ocasion,False,NaN,False,NaN,False,NaN,False,NaN,True,marido de tia,766,776,NaN,NaN,NaN,NaN,NaN,NaN,804.0,818.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,790.0,803.0,NaN,NaN,False,False,0,0
66,961679,ACE_GENERAL_TRAUMA,abusos,False,NaN,False,NaN,False,NaN,True,en una ocasión,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,78,84,NaN,NaN,NaN,NaN,NaN,NaN,132.0,146.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
109,1304145,ACE_SEXUAL_ABUSE,abuso sexual,False,NaN,False,NaN,False,NaN,True,en una sola ocasión,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,128,140,NaN,NaN,NaN,NaN,NaN,NaN,162.0,181.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
110,1304145,ACE_GENERAL_TRAUMA,maltrato,False,NaN,False,NaN,False,NaN,True,en una sola ocasión,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,202,210,NaN,NaN,NaN,NaN,NaN,NaN,162.0,181.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [106]:
# Look for what is recover in childhood_term"
df_filtered = annotations_df[annotations_df["childhood_cue"] == True]
df_filtered.drop(columns=["ehr_history"])



,ehr_id,concept_id,span,negated,negation_term,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,False,NaN,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,206,221,NaN,NaN,222.0,236.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,False,NaN,True,desde pequeña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,867,883,NaN,NaN,885.0,898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
5,232165,ACE_GENERAL_TRAUMA,maltrato,False,NaN,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1894,1902,NaN,NaN,1903.0,1917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
7,262678,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,False,NaN,True,con 6 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,508,524,NaN,NaN,525.0,535.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
8,269487,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,False,NaN,True,durante su infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,423,439,NaN,NaN,440.0,459.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
9,269487,ACE_GENERAL_TRAUMA,maltrato,False,NaN,True,de niña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,728,736,NaN,NaN,757.0,764.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,751.0,756.0,NaN,NaN,False,False,0,0
11,271022,ACE_SEXUAL_ABUSE,abuso sexual,False,NaN,True,cuando tenía 15 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,amigo,83,95,NaN,NaN,106.0,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,105.0,NaN,NaN,False,False,0,0
24,389030,ACE_SEXUAL_ABUSE,abuso sexual,False,NaN,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1419,1431,NaN,NaN,1443.0,1460.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
25,389030,ACE_GENERAL_TRAUMA,maltrato,False,NaN,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,1434,1442,NaN,NaN,1443.0,1460.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1511.0,1516.0,NaN,NaN,False,False,0,0
27,448577,ACE_GENERAL_TRAUMA,abusos,False,NaN,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1263,1269,NaN,NaN,1281.0,1298.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [107]:
# Look for what is recover in duration_term"
print(len(df_filtered))

36


In [108]:
import re

text = "Abusos sex por parte de marido de tia en una ocasion, ..."
pattern = re.compile(r"en\s+una\s+(?:sola\s+|[uú]nica\s+)?ocasi[oó]n", re.I)

print(pattern.findall(text))

['en una ocasion']


### Normalization

In [51]:
normalized_cols_order = [
        "ehr_id",
        "concept_id",
        "span",
        "negation",
        "childhood",
        "adult",
        "temporality",
        "frequency",
        "perpetrator",
        "negated",
        "childhood_cue",
        "childhood_term",
        "adult_cue",
        "adult_term",
        "past_t_cue",
        "past_t_term",
        "recent_cue",
        "recent_term",
        "current_cue",
        "current_term",
        "ongoing_cue",
        "ongoing_term",
        "frequency_cue",
        "frequency_term",
        "perpetrator_cue",
        "perpetrator_term",
        "ehr_history",
    ] + offset_cols + coord_cols

In [52]:
len(annotations_df)

213

In [53]:
normalized_df = normalize_annotations_df(annotations_df, normalized_cols_order, debug=False)

In [54]:
normalized_df.drop(columns=["ehr_history"]).head(10)

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
0,216535,ACE_GENERAL_TRAUMA,victima,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre de su hijo,148,155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.0,202.0,NaN,NaN,False,False,0,0
1,216535,ACE_PHYSICAL_ABUSE,malos tratos,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre de su hijo,159,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.0,202.0,NaN,NaN,False,False,0,0
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,206,221,NaN,NaN,222.0,236.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,false,true,false,past,unspecified,unspecified,False,True,desde pequeña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,867,883,NaN,NaN,885.0,898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
4,216535,ACE_UNSPECIFIED_NEGLECT,negligencia,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1102,1113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
5,232165,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1894,1902,NaN,NaN,1903.0,1917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
6,262678,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,153,161,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
7,262678,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,con 6 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,508,524,NaN,NaN,525.0,535.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
8,269487,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,durante su infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,423,439,NaN,NaN,440.0,459.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
9,269487,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,family_member,False,True,de niña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,728,736,NaN,NaN,757.0,764.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,751.0,756.0,NaN,NaN,False,False,0,0


In [55]:
normalized_df.to_excel("normalized_annotations_ace_confirmed_cases_v3.xlsx", index=False)

#### Test normalized sentences

In [56]:
annotations_df["ehr_id"].unique()

array([  216535,   232165,   262678,   269487,   271022,   305129,
         349669,   365985,   386053,   389030,   403205,   448577,
         464272,   495492,   504137,   564433,   595962,   649848,
         703682,   706171,   784004,   796589,   810687,   814726,
         916274,   923573,   941569,   950325,   955376,   961679,
         976166,   999543,  1025422,  1042052,  1055670,  1072754,
        1077142,  1086522,  1087796,  1113996,  1140733,  1156968,
        1171351,  1202384,  1206473,  1225585,  1227245,  1304145,
        1309521,  1311540,  1332221,  1346534,  1364855,  1369635,
        1375166,  1390426,  1391835,  1461127,  1469042,  1470079,
        1479292,  1492571,  1495588,  1508243,  1518800,  1559801,
        1858236,  2100345,  2173316,  2366090,  4041621,  4072292,
       15036828, 20093050, 20283167, 20354316, 20384566, 20387955,
       22083896, 22190123, 22190230, 22217146, 22301469, 22394845,
       23038503, 24021942, 24087626, 24111281, 25745955], dtyp

In [57]:
note0="Episodio depresivo tras fallecimiento de hijo hace 20 años Refiere maltrato en la infancia"
normalized_note0 = annotate_and_normalize_ace(
    note0,
    nlp,
    phrase_matcher,
    token_matcher
)

normalized_note0

{'annotations': [{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'maltrato',
   'ace_start_char': 67,
   'ace_end_char': 75,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'en la infancia',
   'childhood_start_char': 76,
   'childhood_end_char': 90,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_ch

In [109]:
# Example with two abuses, only one in childhood

normalized_note1 = annotate_and_normalize_ace(
    note1,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note1)
print("Normalized note:")   
normalized_note1

Original note:
haber sido victima de malos tratos por parte del padre de su hijo, y abusos sexuales en la infancia.
Normalized note:


{'annotations': [{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'victima',
   'ace_start_char': 11,
   'ace_end_char': 18,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': Non

In [110]:
# Example with several incidents and perpetrators, and frequency cues
normalized_note2 = annotate_and_normalize_ace(
    note2,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note2)
print("Normalized note:")   
normalized_note2

Original note:
Refiere maltrato por parte del padre de niña. Abusos sex por parte de marido de tia en una ocasion,...
Normalized note:


{'annotations': [{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'maltrato',
   'ace_start_char': 8,
   'ace_end_char': 16,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'de niña',
   'childhood_start_char': 37,
   'childhood_end_char': 44,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': Non

In [111]:
# Examples with "intento de abuso" instead of proper abuso
normalized_note3 = annotate_and_normalize_ace(
    note3,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note3)
print("Normalized note:")   
normalized_note3

Original note:
intento de abuso sexual por su abuelo a los 6 a. No tuvo mucha 
Normalized note:


{'annotations': [{'concept_id': 'ACE_ATTEMPTED_ABUSE',
   'span': 'intento de abuso sexual',
   'ace_start_char': 0,
   'ace_end_char': 23,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'a los 6 a',
   'childhood_start_char': 38,
   'childhood_end_char': 47,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'freque

In [112]:
normalized_note4 = annotate_and_normalize_ace(
    note4,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note4)
print("Normalized note:")   
normalized_note4

Original note:
Intento de abuso sexual del padre con Carmen.
Normalized note:


{'annotations': [{'concept_id': 'ACE_ATTEMPTED_ABUSE',
   'span': 'Intento de abuso sexual',
   'ace_start_char': 0,
   'ace_end_char': 23,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequenc

In [113]:
# Example with post-production, do not consider "abandono" in the context of substance use
normalized_note5 = annotate_and_normalize_ace(
    note5,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note5)
print("Normalized note:")   
normalized_note5

Original note:
Inicio de consumo de alcohol a los 15 años y abandono a los 34 años
Normalized note:


{'annotations': [],
 'coord_postprocessed': False,
 'n_coord_added': 0,
 'n_final_coord_annotations': 0}

In [114]:
# Example with childhood True
normalized_note6 = annotate_and_normalize_ace(
    note6,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note6)
print("Normalized note:")   
normalized_note6

Original note:
Los abusos ocurrieron cuando Joe tenía 6 años de edad
Normalized note:


{'annotations': [{'concept_id': 'ACE_GENERAL_TRAUMA',
   'span': 'abusos',
   'ace_start_char': 4,
   'ace_end_char': 10,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'cuando Joe tenía 6 años',
   'childhood_start_char': 22,
   'childhood_end_char': 45,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_

In [115]:
# Example with conjugated verb (not directly on ACE term dictionary)
normalized_note7 = annotate_and_normalize_ace(
    note7,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note7)
print("Normalized note:")   
normalized_note7

Original note:
Comenta que su hijo, cuando tenía 7 años de edad, fue abusado sexualmente por un vecino.
Normalized note:


{'annotations': [{'concept_id': 'ACE_SEXUAL_ABUSE',
   'span': 'abusado sexualmente',
   'ace_start_char': 54,
   'ace_end_char': 73,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': True,
   'childhood_term': 'cuando tenía 7 años de edad',
   'childhood_start_char': 21,
   'childhood_end_char': 48,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None

In [116]:
normalized_note= annotate_and_normalize_ace(
    note,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note)
print("Normalized note:")   
normalized_note

Original note:
maltrato físico y psicológico en el seno familiar por parte de su madre y una hermana
Normalized note:


{'annotations': [{'concept_id': 'ACE_PHYSICAL_ABUSE',
   'span': 'maltrato físico',
   'ace_start_char': 0,
   'ace_end_char': 15,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_cha

In [117]:
# Example with negation cue
normalized_note8= annotate_and_normalize_ace(
    note8,
    nlp,
    phrase_matcher,
    token_matcher
)

print("Original note:")
print(note8)
print("Normalized note:")   
normalized_note8

Original note:
hubo tocamientos, pero no penetración
Normalized note:


{'annotations': [{'concept_id': 'ACE_SEXUAL_ABUSE',
   'span': 'tocamientos',
   'ace_start_char': 5,
   'ace_end_char': 16,
   'negated': False,
   'negation_term': None,
   'negation_start_char': None,
   'negation_end_char': None,
   'childhood_cue': False,
   'childhood_term': None,
   'childhood_start_char': None,
   'childhood_end_char': None,
   'adult_cue': False,
   'adult_term': None,
   'adult_start_char': None,
   'adult_end_char': None,
   'past_t_cue': False,
   'past_t_term': None,
   'past_t_start_char': None,
   'past_t_end_char': None,
   'recent_cue': False,
   'recent_term': None,
   'recent_start_char': None,
   'recent_end_char': None,
   'current_cue': False,
   'current_term': None,
   'current_start_char': None,
   'current_end_char': None,
   'ongoing_cue': False,
   'ongoing_term': None,
   'ongoing_start_char': None,
   'ongoing_end_char': None,
   'frequency_cue': False,
   'frequency_term': None,
   'frequency_start_char': None,
   'frequency_end_char': No

In [118]:
# Examine results for a given EHR in the normalized dataframe
# Examine a given EHR 
ehr_id = 1492571
row = normalized_df.loc[annotations_df["ehr_id"] == ehr_id]
print(row["ehr_history"].iloc[0])

ÚNICA VALORACIÓN EN SEPTIEMBRE DE 2014:(DRA.POZA)
Comenta que su hijo, cuando tenía 7 años de edad, fue abusado sexualmente por un vecino.
-AP:
-Familia de origen rumano. Los padres se vinieron a España hace 8 años y Radu se vino hace 7 años. Padres separados desde hace 3 años. Radu vive con la madre y lleva un año sin ver al padre (el padre reside en Rumanía desde hace un año).
La madre refiere maltrato psicológico por parte del padre de Radu; en una ocasión, estando ya separados, agredió físicamente a la madre.
-No AP médicos de interés.
-Va a cursar 6º de Primaria en el CEIP "Antonio Machado". Repitió 2º de Primaria.
-Los abusos ocurrieron cuando Radu tenía 6 años de edad; vivía en Rumanía, a cargo de los abuelos. El abusador era un vecino, tenía 80 años de edad; hubo tocamientos, pero no penetración. Se denunció, pero alegaron enajenación mental y no hubo condena.
-MC: Abuso sexual.
El paciente ha conseguido hablar todos estos temas con su madre; le pregunta que si es gay, que se s

In [119]:
row.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
138,1492571,ACE_SEXUAL_ABUSE,abusado sexualmente,false,true,false,past,unspecified,known_other,False,True,cuando tenía 7 años de edad,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,vecino,104,123,NaN,NaN,71.0,98.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131.0,137.0,NaN,NaN,False,False,0,0
139,1492571,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,unspecified,unspecified,ongoing,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,desde hace 3 años,False,NaN,False,NaN,243,259,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260.0,277.0,NaN,NaN,NaN,NaN,False,False,0,0
140,1492571,ACE_EMOTIONAL_ABUSE,maltrato psicológico,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,399,419,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,434.0,439.0,NaN,NaN,False,False,0,0
141,1492571,ACE_GENERAL_TRAUMA,abusos,false,true,false,past,unspecified,unspecified,False,True,cuando Radu tenía 6 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,633,639,NaN,NaN,651.0,675.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
142,1492571,ACE_SEXUAL_ABUSE,tocamientos,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,782,793,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
143,1492571,ACE_SEXUAL_ABUSE,penetración,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,803,814,800.0,803.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
144,1492571,ACE_SEXUAL_ABUSE,Abuso sexual,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,886,898,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [120]:
normalized_df.columns

Index(['ehr_id', 'concept_id', 'span', 'negation', 'childhood', 'adult',
       'temporality', 'frequency', 'perpetrator', 'negated', 'childhood_cue',
       'childhood_term', 'adult_cue', 'adult_term', 'past_t_cue',
       'past_t_term', 'recent_cue', 'recent_term', 'current_cue',
       'current_term', 'ongoing_cue', 'ongoing_term', 'frequency_cue',
       'frequency_term', 'perpetrator_cue', 'perpetrator_term', 'ehr_history',
       'ace_start_char', 'ace_end_char', 'negation_start_char',
       'negation_end_char', 'childhood_start_char', 'childhood_end_char',
       'adult_start_char', 'adult_end_char', 'frequency_start_char',
       'frequency_end_char', 'past_t_start_char', 'past_t_end_char',
       'recent_start_char', 'recent_end_char', 'current_start_char',
       'current_end_char', 'ongoing_start_char', 'ongoing_end_char',
       'perpetrator_start_char', 'perpetrator_end_char', 'expanded_head',
       'normalized_span', 'inferred_from_coordination', 'coord_postprocessed',


##### Test the recovery of certain unusual attributes

In [121]:
# Look for what is recover in "recent_term"
df_ann_filtered = normalized_df[normalized_df["temporality"] == "recent"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations


In [122]:
# Look for what is recover in "current_cue" 
df_ann_filtered = normalized_df[normalized_df["temporality"] == "current"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
184,22190123,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,current,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,True,en la actualidad,False,NaN,False,NaN,False,NaN,2816,2824,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2841.0,2857.0,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
187,22190123,ACE_GENERAL_TRAUMA,violencia,false,unspecified,unspecified,current,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,True,actualmente,False,NaN,False,NaN,False,NaN,5284,5293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5250.0,5261.0,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [123]:
# Look for what is recover in "past_t_term"
df_ann_filtered = normalized_df[normalized_df["temporality"] == "past"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,206,221,NaN,NaN,222.0,236.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,false,true,false,past,unspecified,unspecified,False,True,desde pequeña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,867,883,NaN,NaN,885.0,898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
5,232165,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1894,1902,NaN,NaN,1903.0,1917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
7,262678,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,con 6 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,508,524,NaN,NaN,525.0,535.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
8,269487,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,durante su infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,423,439,NaN,NaN,440.0,459.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
9,269487,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,family_member,False,True,de niña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,728,736,NaN,NaN,757.0,764.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,751.0,756.0,NaN,NaN,False,False,0,0
11,271022,ACE_SEXUAL_ABUSE,abuso sexual,false,true,false,past,unspecified,known_other,False,True,cuando tenía 15 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,amigo,83,95,NaN,NaN,106.0,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,105.0,NaN,NaN,False,False,0,0
24,389030,ACE_SEXUAL_ABUSE,abuso sexual,false,true,false,past,unspecified,unspecified,False,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1419,1431,NaN,NaN,1443.0,1460.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
25,389030,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,family_member,False,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,1434,1442,NaN,NaN,1443.0,1460.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1511.0,1516.0,NaN,NaN,False,False,0,0
27,448577,ACE_GENERAL_TRAUMA,abusos,false,true,false,past,unspecified,unspecified,False,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1263,1269,NaN,NaN,1281.0,1298.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [124]:
# Look for what is recover in "ongoing_term"
df_ann_filtered = normalized_df[normalized_df["temporality"] == "ongoing"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
139,1492571,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,unspecified,unspecified,ongoing,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,desde hace 3 años,False,NaN,False,NaN,243,259,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260.0,277.0,NaN,NaN,NaN,NaN,False,False,0,0


In [125]:
# Look for what is recover in "unsècidfied" temporality 
df_ann_filtered = normalized_df[normalized_df["temporality"] == "unspecified"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
0,216535,ACE_GENERAL_TRAUMA,victima,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre de su hijo,148,155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.0,202.0,NaN,NaN,False,False,0,0
1,216535,ACE_PHYSICAL_ABUSE,malos tratos,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre de su hijo,159,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.0,202.0,NaN,NaN,False,False,0,0
4,216535,ACE_UNSPECIFIED_NEGLECT,negligencia,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1102,1113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
6,262678,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,153,161,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
10,269487,ACE_SEXUAL_ABUSE,Abusos sex,false,unspecified,unspecified,unspecified,single,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,en una ocasion,True,marido de tia,766,776,NaN,NaN,NaN,NaN,NaN,NaN,804.0,818.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,790.0,803.0,NaN,NaN,False,False,0,0
12,271022,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,384,392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
13,305129,ACE_PHYSICAL_ABUSE,maltrato físico,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,madre,3431,3446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3497.0,3502.0,NaN,NaN,False,True,1,1
14,305129,ACE_EMOTIONAL_ABUSE,psicológico,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,madre,3449,3460,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3497.0,3502.0,maltrato,maltrato psicológico,True,True,1,1
15,349669,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,156,164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
16,349669,ACE_GENERAL_TRAUMA,abusos,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,167,173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [126]:
# Look for what is recover in "negation"
df_ann_filtered = normalized_df[normalized_df["negation"] == "true"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
76,1055670,ACE_GENERAL_TRAUMA,abusos,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,210,216,191.0,209.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
77,1055670,ACE_GENERAL_TRAUMA,maltrato,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,220,228,191.0,209.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
104,1225585,ACE_GENERAL_TRAUMA,maltrato,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,2110,2118,2104.0,2109.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
127,1391835,ACE_PHYSICAL_ABUSE,maltrato físico,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,3844,3859,3825.0,3830.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
143,1492571,ACE_SEXUAL_ABUSE,penetración,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,803,814,800.0,803.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
172,15036828,ACE_GENERAL_TRAUMA,maltrato,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,923,931,905.0,910.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
202,22394845,ACE_PHYSICAL_ABUSE,maltrato físico,true,unspecified,unspecified,unspecified,unspecified,unspecified,True,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,2585,2600,2581.0,2584.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [127]:
# Look for what is recover in "childhood" True
df_ann_filtered = normalized_df[normalized_df["childhood"] == "true"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,206,221,NaN,NaN,222.0,236.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,false,true,false,past,unspecified,unspecified,False,True,desde pequeña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,867,883,NaN,NaN,885.0,898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
5,232165,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1894,1902,NaN,NaN,1903.0,1917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
7,262678,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,con 6 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,508,524,NaN,NaN,525.0,535.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
8,269487,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,durante su infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,423,439,NaN,NaN,440.0,459.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
9,269487,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,family_member,False,True,de niña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,728,736,NaN,NaN,757.0,764.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,751.0,756.0,NaN,NaN,False,False,0,0
11,271022,ACE_SEXUAL_ABUSE,abuso sexual,false,true,false,past,unspecified,known_other,False,True,cuando tenía 15 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,amigo,83,95,NaN,NaN,106.0,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,105.0,NaN,NaN,False,False,0,0
24,389030,ACE_SEXUAL_ABUSE,abuso sexual,false,true,false,past,unspecified,unspecified,False,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1419,1431,NaN,NaN,1443.0,1460.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
25,389030,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,family_member,False,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,1434,1442,NaN,NaN,1443.0,1460.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1511.0,1516.0,NaN,NaN,False,False,0,0
27,448577,ACE_GENERAL_TRAUMA,abusos,false,true,false,past,unspecified,unspecified,False,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1263,1269,NaN,NaN,1281.0,1298.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [128]:
# Look for what is recover in "childhood" False
df_ann_filtered = normalized_df[normalized_df["childhood"] == "false"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
171,4072292,ACE_PHYSICAL_ABUSE,maltrato físico,false,false,true,unspecified,unspecified,unspecified,False,False,NaN,True,a los 31 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,707,722,NaN,NaN,NaN,NaN,690.0,703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
176,20283167,ACE_GENERAL_TRAUMA,maltrato,false,false,true,unspecified,unspecified,partner,False,False,NaN,True,a los 26 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,58,66,NaN,NaN,NaN,NaN,40.0,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,84.0,90.0,NaN,NaN,False,False,0,0


In [129]:
# Look for what is recover in "adulthood" True
df_ann_filtered = normalized_df[normalized_df["adult"] == "true"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
171,4072292,ACE_PHYSICAL_ABUSE,maltrato físico,false,false,true,unspecified,unspecified,unspecified,False,False,NaN,True,a los 31 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,707,722,NaN,NaN,NaN,NaN,690.0,703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
176,20283167,ACE_GENERAL_TRAUMA,maltrato,false,false,true,unspecified,unspecified,partner,False,False,NaN,True,a los 26 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,58,66,NaN,NaN,NaN,NaN,40.0,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,84.0,90.0,NaN,NaN,False,False,0,0


In [130]:
# Look for what is recover in "single" True
df_ann_filtered = normalized_df[normalized_df["frequency"] == "single"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
10,269487,ACE_SEXUAL_ABUSE,Abusos sex,false,unspecified,unspecified,unspecified,single,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,en una ocasion,True,marido de tia,766,776,NaN,NaN,NaN,NaN,NaN,NaN,804.0,818.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,790.0,803.0,NaN,NaN,False,False,0,0
66,961679,ACE_GENERAL_TRAUMA,abusos,false,unspecified,unspecified,unspecified,single,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,en una ocasión,False,NaN,78,84,NaN,NaN,NaN,NaN,NaN,NaN,132.0,146.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
109,1304145,ACE_SEXUAL_ABUSE,abuso sexual,false,unspecified,unspecified,unspecified,single,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,en una sola ocasión,False,NaN,128,140,NaN,NaN,NaN,NaN,NaN,NaN,162.0,181.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
110,1304145,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,single,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,en una sola ocasión,False,NaN,202,210,NaN,NaN,NaN,NaN,NaN,NaN,162.0,181.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0


In [131]:
# Look for what is recover in "repeated" True
df_ann_filtered = normalized_df[normalized_df["frequency"] == "repeated"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations


In [132]:
# Look for what is recover in "recurrent" True
df_ann_filtered = normalized_df[normalized_df["frequency"] == "recurrent"]
df_ann_filtered.drop(columns=["ehr_history"])

,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations


In [133]:
# Look for what is recover in "perpetrator" family_member
df_ann_filtered = normalized_df[normalized_df["perpetrator"] == "family_member"]
print(len(df_ann_filtered))
df_ann_filtered.drop(columns=["ehr_history"])

32


,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
0,216535,ACE_GENERAL_TRAUMA,victima,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre de su hijo,148,155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.0,202.0,NaN,NaN,False,False,0,0
1,216535,ACE_PHYSICAL_ABUSE,malos tratos,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre de su hijo,159,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,186.0,202.0,NaN,NaN,False,False,0,0
9,269487,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,family_member,False,True,de niña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,728,736,NaN,NaN,757.0,764.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,751.0,756.0,NaN,NaN,False,False,0,0
10,269487,ACE_SEXUAL_ABUSE,Abusos sex,false,unspecified,unspecified,unspecified,single,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,en una ocasion,True,marido de tia,766,776,NaN,NaN,NaN,NaN,NaN,NaN,804.0,818.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,790.0,803.0,NaN,NaN,False,False,0,0
13,305129,ACE_PHYSICAL_ABUSE,maltrato físico,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,madre,3431,3446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3497.0,3502.0,NaN,NaN,False,True,1,1
14,305129,ACE_EMOTIONAL_ABUSE,psicológico,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,madre,3449,3460,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3497.0,3502.0,maltrato,maltrato psicológico,True,True,1,1
25,389030,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,family_member,False,True,desde la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,padre,1434,1442,NaN,NaN,1443.0,1460.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1511.0,1516.0,NaN,NaN,False,False,0,0
35,564433,ACE_ATTEMPTED_ABUSE,intento de abuso sexual,false,true,false,past,unspecified,family_member,False,True,a los 6 a,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,abuelo,171,194,NaN,NaN,209.0,218.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202.0,208.0,NaN,NaN,False,False,0,0
40,703682,ACE_SEXUAL_ABUSE,abuso sexual,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,prima,552,564,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,582.0,587.0,NaN,NaN,False,False,0,0
41,703682,ACE_EMOTIONAL_ABUSE,abuso psicologico,false,unspecified,unspecified,unspecified,unspecified,family_member,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,prima,590,607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,582.0,587.0,NaN,NaN,False,False,0,0


In [134]:
# Look for what is recover in "perpetrator" family_member
df_ann_filtered = normalized_df[normalized_df["perpetrator"] == "partner"]
print(len(df_ann_filtered))
df_ann_filtered.drop(columns=["ehr_history"])

11


,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
19,365985,ACE_PHYSICAL_ABUSE,maltrato físico,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,ex,580,595,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,626.0,628.0,NaN,NaN,False,True,1,1
20,365985,ACE_EMOTIONAL_ABUSE,psicológico,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,ex,598,609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,626.0,628.0,maltrato,maltrato psicológico,True,True,1,1
44,706171,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,311,319,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,336.0,342.0,NaN,NaN,False,False,0,0
47,784004,ACE_SEXUAL_ABUSE,abusos sexuales,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,143,158,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,215.0,221.0,NaN,NaN,False,False,0,0
48,784004,ACE_PHYSICAL_ABUSE,maltrato físico,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,161,176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,215.0,221.0,NaN,NaN,False,False,0,0
67,961679,ACE_PHYSICAL_ABUSE,maltrato físico,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,ex,958,973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,981.0,983.0,NaN,NaN,False,False,0,0
71,999543,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,marido,623,631,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,648.0,654.0,NaN,NaN,False,False,0,0
83,1087796,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,93,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.0,88.0,NaN,NaN,False,False,0,0
152,1518800,ACE_PHYSICAL_ABUSE,maltrato físico,false,unspecified,unspecified,unspecified,unspecified,partner,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,318,333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,351.0,357.0,NaN,NaN,False,False,0,0
176,20283167,ACE_GENERAL_TRAUMA,maltrato,false,false,true,unspecified,unspecified,partner,False,False,NaN,True,a los 26 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,pareja,58,66,NaN,NaN,NaN,NaN,40.0,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,84.0,90.0,NaN,NaN,False,False,0,0


In [135]:
# Look for what is recover in "perpetrator" known_other
df_ann_filtered = normalized_df[normalized_df["perpetrator"] == "known_other"]
print(len(df_ann_filtered))
df_ann_filtered.drop(columns=["ehr_history"])

3


,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
11,271022,ACE_SEXUAL_ABUSE,abuso sexual,false,true,false,past,unspecified,known_other,False,True,cuando tenía 15 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,amigo,83,95,NaN,NaN,106.0,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,105.0,NaN,NaN,False,False,0,0
72,1025422,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,known_other,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,profesora,296,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,322.0,331.0,NaN,NaN,False,False,0,0
138,1492571,ACE_SEXUAL_ABUSE,abusado sexualmente,false,true,false,past,unspecified,known_other,False,True,cuando tenía 7 años de edad,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,True,vecino,104,123,NaN,NaN,71.0,98.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131.0,137.0,NaN,NaN,False,False,0,0


In [136]:
# Look for what is recover in "perpetrator" unknown_other
df_ann_filtered = normalized_df[normalized_df["perpetrator"] == "unknown_other"]
print(len(df_ann_filtered))
df_ann_filtered.drop(columns=["ehr_history"])

0


,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations


In [137]:
# Look for what is recover in "perpetrator" unspecified
df_ann_filtered = normalized_df[normalized_df["perpetrator"] == "unspecified"]
print(len(df_ann_filtered))
df_ann_filtered.drop(columns=["ehr_history"])

167


,ehr_id,concept_id,span,negation,childhood,adult,temporality,frequency,perpetrator,negated,childhood_cue,childhood_term,adult_cue,adult_term,past_t_cue,past_t_term,recent_cue,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,frequency_cue,frequency_term,perpetrator_cue,perpetrator_term,ace_start_char,ace_end_char,negation_start_char,negation_end_char,childhood_start_char,childhood_end_char,adult_start_char,adult_end_char,frequency_start_char,frequency_end_char,past_t_start_char,past_t_end_char,recent_start_char,recent_end_char,current_start_char,current_end_char,ongoing_start_char,ongoing_end_char,perpetrator_start_char,perpetrator_end_char,expanded_head,normalized_span,inferred_from_coordination,coord_postprocessed,n_coord_added,n_final_coord_annotations
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,206,221,NaN,NaN,222.0,236.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,false,true,false,past,unspecified,unspecified,False,True,desde pequeña,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,867,883,NaN,NaN,885.0,898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
4,216535,ACE_UNSPECIFIED_NEGLECT,negligencia,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1102,1113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
5,232165,ACE_GENERAL_TRAUMA,maltrato,false,true,false,past,unspecified,unspecified,False,True,en la infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,1894,1902,NaN,NaN,1903.0,1917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
6,262678,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,153,161,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
7,262678,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,con 6 años,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,508,524,NaN,NaN,525.0,535.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
8,269487,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,false,true,false,past,unspecified,unspecified,False,True,durante su infancia,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,423,439,NaN,NaN,440.0,459.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
12,271022,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,384,392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
15,349669,ACE_GENERAL_TRAUMA,maltrato,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,156,164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
16,349669,ACE_GENERAL_TRAUMA,abusos,false,unspecified,unspecified,unspecified,unspecified,unspecified,False,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN,167,173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0
